In [3]:
!cd "/Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON" && \
SYNTHETIC_COUNT=20 FACTURE_TEMPLATE=all python3 generate_facture_images.py

zsh:cd:1: no such file or directory: /Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON


## PaddleOCR facture pipeline (clean)

Notebook nettoyé : on garde uniquement le pipeline **PaddleOCR** + détection/champs.
Ce bloc définit les variables utilitaires utilisées plus loin (`FACTURE_IMG_DIR`, `list_facture_images`).


In [4]:
from pathlib import Path

# Dossier des images factures
FACTURE_IMG_DIR = "/Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/ocr traitement/facture-images"

def list_facture_images(folder: str, pattern: str = None):
    """Liste les images facture (jpg/png) dans un dossier."""
    p = Path(folder)
    if not p.exists():
        return []
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        files.extend(sorted(p.glob(ext)))
    out = [str(f) for f in files]
    if pattern:
        out = [f for f in out if pattern.lower() in Path(f).name.lower()]
    return out

print("FACTURE_IMG_DIR:", FACTURE_IMG_DIR)
print("Found images:", len(list_facture_images(FACTURE_IMG_DIR)))


FACTURE_IMG_DIR: /Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/ocr traitement/facture-images
Found images: 0


## Pipeline idéal — PaddleOCR pour factures

**Outils utilisés :**
- **Layout detection** : découpage en zones (header, tableau, total)
- **PaddleOCR** : très performant pour les factures (OCR par zone)
- *Alternatives possibles* : LayoutLM, Donut (Naver) pour un pipeline encore plus fin

**Étapes :**
1. Détection des zones (layout)
2. Découper : header → tableau → total
3. OCR sur chaque bloc
4. Afficher chaque découpe et le texte reconnu


### Installation PaddleOCR

Exécuter **une fois**. Si la cellule bloque ou erre après Ctrl+C, lancez dans un **terminal** : `pip install paddleocr` puis redémarrez le noyau. Sur Mac M1/M2, `paddlepaddle` est optionnel.


In [5]:
# Install PaddleOCR (run ONCE)
# Uses subprocess to avoid IPython/pexpect KeyboardInterrupt bug with !pip
import subprocess
import sys
for pkg in ["paddleocr", "paddlepaddle"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], capture_output=True, text=True)
    if r.returncode != 0 and pkg == "paddlepaddle":
        print("paddlepaddle skipped (optional; use CPU or install manually).")
print("PaddleOCR install done. Restart kernel if needed.")


PaddleOCR install done. Restart kernel if needed.


In [6]:
# test_images = list_facture_images(FACTURE_IMG_DIR, pattern="clean")

# if not test_images:
#     print("No facture images found in:", FACTURE_IMG_DIR)

# else:
#     test_path = test_images[0]
#     print("Testing TrOCR on:", test_path)

#     try:
#         text = trocr_ocr_image_to_text(test_path)

#         print("\n----- OCR RESULT -----\n")
#         print(text[:1500])

#     except Exception as e:
#         print("Error while running TrOCR test:", e)
pass

### Initialisation PaddleOCR (cellule dédiée)

À exécuter après l’installation. Cette cellule initialise `paddle_ocr` et définit des helpers pour parser les sorties (`predict()`), afin que les cellules Header/Tableau/Total fonctionnent quel que soit le format de retour.

In [7]:
import os
import numpy as np

# Évite la vérification de connexion aux hébergeurs de modèles
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

try:
    from paddleocr import PaddleOCR

    # Compat: certaines versions utilisent use_textline_orientation, d'autres use_angle_cls
    try:
        paddle_ocr = PaddleOCR(use_textline_orientation=True, lang="fr")
        print("[PaddleOCR] Initialisé (lang=fr, use_textline_orientation=True).")
    except TypeError:
        paddle_ocr = PaddleOCR(use_angle_cls=True, lang="fr")
        print("[PaddleOCR] Initialisé (lang=fr, use_angle_cls=True).")
except Exception as e:
    paddle_ocr = None
    print("[PaddleOCR] Non disponible:", e)


def _paddle_result_to_text(result):
    """Extrait un texte lisible depuis paddle_ocr.predict() (format v2/v3, caractères ou lignes)."""
    if not result:
        return "(vide)"
    parts = []
    r0 = result[0] if isinstance(result, (list, tuple)) else result

    if isinstance(r0, dict):
        rec = r0.get("rec_texts", r0.get("text", []))
        if isinstance(rec, list):
            for item in rec:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, (list, tuple)):
                    parts.extend([str(x) for x in item if x])
    elif isinstance(r0, (list, tuple)):
        for line in r0:
            if isinstance(line, (list, tuple)) and len(line) >= 2:
                text_part = line[1]
                if isinstance(text_part, (list, tuple)):
                    parts.append(str(text_part[0]))
                else:
                    parts.append(str(text_part))
            elif isinstance(line, str):
                parts.append(line)

    if not parts:
        return "(vide)"

    # Si beaucoup de tokens d'un seul caractère, joindre en une ligne
    if sum(1 for p in parts if len(p) <= 1) > len(parts) / 2:
        return " ".join(parts)

    return "\n".join(parts)


def _paddle_predict_to_items(result):
    """Liste d'items {box, text, score} depuis paddle_ocr.predict()."""
    items = []
    if not result:
        return items
    r0 = result[0] if isinstance(result, (list, tuple)) else result

    if isinstance(r0, dict):
        texts = r0.get("rec_texts") or r0.get("texts") or r0.get("text") or []
        scores = r0.get("rec_scores") or r0.get("scores") or []
        boxes = r0.get("dt_polys") or r0.get("dt_boxes") or r0.get("boxes") or r0.get("polys") or []
        n = min(len(texts), len(boxes)) if texts and boxes else 0
        for i in range(n):
            items.append({"box": np.array(boxes[i]), "text": str(texts[i]), "score": scores[i] if i < len(scores) else None})
        return items

    if isinstance(r0, (list, tuple)):
        for det in r0:
            if isinstance(det, (list, tuple)) and len(det) >= 2:
                box = np.array(det[0])
                rec = det[1]
                if isinstance(rec, (list, tuple)) and len(rec) >= 1:
                    text = rec[0]
                    score = rec[1] if len(rec) >= 2 else None
                else:
                    text = rec
                    score = None
                items.append({"box": box, "text": str(text), "score": score})
        return items

    return items

Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached

[PaddleOCR] Initialisé (lang=fr, use_textline_orientation=True).


### Imports et initialisation PaddleOCR

Modèle français, détection d'angle activée (idéal pour factures).


In [8]:
# Imports + init PaddleOCR
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

# Évite la vérification de connexion aux hébergeurs de modèles
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# Init PaddleOCR (lang fr pour factures françaises)
try:
    from paddleocr import PaddleOCR
    paddle_ocr = PaddleOCR(use_textline_orientation=True, lang="fr")
    print("[PaddleOCR] Initialisé (lang=fr, use_textline_orientation=True).")
except Exception as e:
    paddle_ocr = None
    print("[PaddleOCR] Non disponible:", e)

def _paddle_result_to_text(result):
    """Extrait le texte depuis paddle_ocr.predict() (format v2/v3, caractères ou lignes)."""
    if not result:
        return "(vide)"
    parts = []
    r0 = result[0] if isinstance(result, (list, tuple)) else result
    if isinstance(r0, dict):
        rec = r0.get("rec_texts", r0.get("text", []))
        if isinstance(rec, list):
            for item in rec:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, (list, tuple)):
                    parts.extend([str(x) for x in item if x])
    elif isinstance(r0, (list, tuple)):
        for line in r0:
            if isinstance(line, (list, tuple)) and len(line) >= 2:
                text_part = line[1]
                if isinstance(text_part, (list, tuple)):
                    parts.append(str(text_part[0]))
                else:
                    parts.append(str(text_part))
            elif isinstance(line, str):
                parts.append(line)
    if not parts:
        return "(vide)"
    if sum(1 for p in parts if len(p) <= 1) > len(parts) / 2:
        return " ".join(parts)
    return "\n".join(parts)


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/aymanehajli/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/aymanehajli/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/aymanehajli/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/aymanehajli/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('latin_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/aymanehajli/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


[PaddleOCR] Initialisé (lang=fr, use_textline_orientation=True).


### Détection des zones (layout)

On découpe la facture en 3 zones selon la hauteur :
- **Header** : haut (0–25 %) — en-tête, n° facture, dates
- **Tableau** : milieu (25–75 %) — lignes de produits
- **Total** : bas (75–100 %) — totaux HT, TVA, TTC

*Pour un layout plus fin : LayoutLM ou Donut (classification de régions).*


In [9]:
# Charger une facture et définir les 3 zones (layout)
if "FACTURE_IMG_DIR" not in dir():
    FACTURE_IMG_DIR = "/Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/output/facture-images"
FACTURE_EXAMPLE = Path(FACTURE_IMG_DIR) / "facture_2026006_clean.jpg"
if not FACTURE_EXAMPLE.exists():
    FACTURE_EXAMPLE = list(Path(FACTURE_IMG_DIR).glob("*clean*.jpg"))[:1]
    FACTURE_EXAMPLE = FACTURE_EXAMPLE[0] if FACTURE_EXAMPLE else None

if FACTURE_EXAMPLE is None:
    print("Aucune image facture trouvée dans", FACTURE_IMG_DIR)
else:
    img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_bgr.shape[:2]
    # Zones en % de la hauteur
    header_y1, header_y2 = 0, int(0.40 * h)
    tableau_y1, tableau_y2 = int(0.40 * h), int(0.5 * h)
    total_y1, total_y2 = int(0.5 * h), h
    # Dessiner les 3 zones sur l'image
    vis = img_bgr.copy()
    cv2.rectangle(vis, (0, header_y1), (w, header_y2), (0, 255, 0), 3)
    cv2.putText(vis, "HEADER", (10, header_y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.rectangle(vis, (0, tableau_y1), (w, tableau_y2), (255, 0, 0), 3)
    cv2.putText(vis, "TABLEAU", (10, tableau_y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.rectangle(vis, (0, total_y1), (w, total_y2), (0, 0, 255), 3)
    cv2.putText(vis, "TOTAL", (10, total_y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    plt.figure(figsize=(10, 12))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title("Détection des zones (Header / Tableau / Total)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("Image chargée:", FACTURE_EXAMPLE.name)


Aucune image facture trouvée dans /Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/ocr traitement/facture-images


### Zone 1 — Header (image découpée)

Affichage de la zone **header** uniquement.


In [10]:
# Découpe HEADER — afficher l'image
# if FACTURE_EXAMPLE is not None:
#     img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
#     h, w = img_bgr.shape[:2]
#     header_y1, header_y2 = 0, int(0.25 * h)
#     crop_header = img_bgr[header_y1:header_y2, :]
#     plt.figure(figsize=(10, 4))
#     plt.imshow(cv2.cvtColor(crop_header, cv2.COLOR_BGR2RGB))
#     plt.title("Zone HEADER (découpée)")
#     plt.axis("off")
#     plt.tight_layout()
#     plt.show()
pass

### OCR sur le header

PaddleOCR appliqué sur la zone header uniquement.


In [11]:
# OCR sur le bloc HEADER
if paddle_ocr is not None and FACTURE_EXAMPLE is not None:
    img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
    h, w = img_bgr.shape[:2]
    crop_header = img_bgr[0:int(0.25 * h), :]
    result = paddle_ocr.predict(crop_header)
    text_header = _paddle_result_to_text(result)
    print("=== OCR HEADER ===\n")
    print(text_header)
else:
    print("PaddleOCR ou image non disponible.")


PaddleOCR ou image non disponible.


### Zone 2 — Tableau (image découpée)

Affichage de la zone **tableau** (lignes de produits).


In [12]:
# Découpe TABLEAU — afficher l'image
# if FACTURE_EXAMPLE is not None:
#     img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
#     h, w = img_bgr.shape[:2]
#     y1, y2 = int(0.25 * h), int(0.75 * h)
#     crop_tableau = img_bgr[y1:y2, :]
#     plt.figure(figsize=(10, 8))
#     plt.imshow(cv2.cvtColor(crop_tableau, cv2.COLOR_BGR2RGB))
#     plt.title("Zone TABLEAU (découpée)")
#     plt.axis("off")
#     plt.tight_layout()
#     plt.show()
pass

### OCR sur le tableau

PaddleOCR sur la zone tableau.


In [13]:
# OCR sur le bloc TABLEAU
if paddle_ocr is not None and FACTURE_EXAMPLE is not None:
    img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
    h, w = img_bgr.shape[:2]
    crop_tableau = img_bgr[int(0.25 * h):int(0.75 * h), :]
    result = paddle_ocr.predict(crop_tableau)
    text_tableau = _paddle_result_to_text(result)
    print("=== OCR TABLEAU ===\n")
    print(text_tableau[:2000])
    if len(text_tableau) > 2000:
        print("... (tronqué)")
else:
    print("PaddleOCR ou image non disponible.")


PaddleOCR ou image non disponible.


### Zone 3 — Total (image découpée)

Affichage de la zone **total** (totaux HT, TVA, TTC).


In [14]:
# # Découpe TOTAL — afficher l'image
# if FACTURE_EXAMPLE is not None:
#     img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
#     h, w = img_bgr.shape[:2]
#     y1, y2 = int(0.75 * h), h
#     crop_total = img_bgr[y1:y2, :]
#     plt.figure(figsize=(10, 4))
#     plt.imshow(cv2.cvtColor(crop_total, cv2.COLOR_BGR2RGB))
#     plt.title("Zone TOTAL (découpée)")
#     plt.axis("off")
#     plt.tight_layout()
#     plt.show()
pass


### OCR sur le total

PaddleOCR sur la zone total.


In [15]:
# OCR sur le bloc TOTAL
if paddle_ocr is not None and FACTURE_EXAMPLE is not None:
    img_bgr = cv2.imread(str(FACTURE_EXAMPLE))
    h, w = img_bgr.shape[:2]
    crop_total = img_bgr[int(0.75 * h):h, :]
    result = paddle_ocr.predict(crop_total)
    text_total = _paddle_result_to_text(result)
    print("=== OCR TOTAL ===\n")
    print(text_total)
else:
    print("PaddleOCR ou image non disponible.")


PaddleOCR ou image non disponible.


### Résultat combiné (pipeline complet)

Texte final : Header + Tableau + Total.


In [16]:
# Résultat combiné des 3 zones
text_header = globals().get("text_header", "(exécuter la cellule OCR Header)")
text_tableau = globals().get("text_tableau", "(exécuter la cellule OCR Tableau)")
text_total = globals().get("text_total", "(exécuter la cellule OCR Total)")
if paddle_ocr is not None and FACTURE_EXAMPLE is not None:
    full_text = f"""=== HEADER ===
{text_header}

=== TABLEAU ===
{text_tableau}

=== TOTAL ===
{text_total}
"""
    print(full_text)
else:
    print("Exécuter les cellules OCR ci-dessus d'abord.")


Exécuter les cellules OCR ci-dessus d'abord.


### Détection directe des champs sur l’image (avec bounding boxes)

Cette partie :
- lance PaddleOCR sur **l’image complète**
- récupère les **boîtes (bbox) + texte**
- détecte les champs clés (Facture N°, dates, email, tél, SIRET, totaux…)
- **dessine des rectangles + labels** directement sur l’image (comme ton exemple).

In [17]:
import re
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Charger une facture et définir les 3 zones (layout)
if "FACTURE_IMG_DIR" not in dir():
    FACTURE_IMG_DIR = "/Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/output/facture-images"
FACTURE_EXAMPLE = Path(FACTURE_IMG_DIR) / "facture_2026015_boxed_rotation.jpg"
if not FACTURE_EXAMPLE.exists():
    FACTURE_EXAMPLE = list(Path(FACTURE_IMG_DIR).glob("*clean*.jpg"))[:1]
    FACTURE_EXAMPLE = FACTURE_EXAMPLE[0] if FACTURE_EXAMPLE else None

if FACTURE_EXAMPLE is None:
    print("Aucune image facture trouvée dans", FACTURE_IMG_DIR)
def _paddle_predict_to_items(result):
    """Return list of {box, text, score} from PaddleOCR predict() output.
    Supports multiple PaddleOCR versions/shapes.
    """
    items = []
    if not result:
        return items

    # Common: list[0] contains detections
    r0 = result[0] if isinstance(result, (list, tuple)) else result

    # Dict format (v3 pipelines often return dict-like)
    if isinstance(r0, dict):
        texts = r0.get("rec_texts") or r0.get("texts") or r0.get("text") or []
        scores = r0.get("rec_scores") or r0.get("scores") or []
        boxes = (
            r0.get("dt_polys")
            or r0.get("dt_boxes")
            or r0.get("boxes")
            or r0.get("polys")
            or []
        )
        n = min(len(texts), len(boxes)) if texts and boxes else 0
        for i in range(n):
            box = np.array(boxes[i])
            text = texts[i]
            score = scores[i] if i < len(scores) else None
            items.append({"box": box, "text": str(text), "score": score})
        return items

    # List/tuple format: [box, (text, score)]
    if isinstance(r0, (list, tuple)):
        for det in r0:
            if isinstance(det, (list, tuple)) and len(det) >= 2:
                box = np.array(det[0])
                rec = det[1]
                if isinstance(rec, (list, tuple)) and len(rec) >= 1:
                    text = rec[0]
                    score = rec[1] if len(rec) >= 2 else None
                else:
                    text = rec
                    score = None
                items.append({"box": box, "text": str(text), "score": score})
        return items

    return items


def _box_to_rect(box):
    """Convert 4-point poly to (x1,y1,x2,y2)."""
    box = np.array(box)
    xs = box[:, 0]
    ys = box[:, 1]
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def _rect_center(r):
    x1, y1, x2, y2 = r
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def _union_rect(rects, pad=6):
    if not rects:
        return None
    x1 = min(r[0] for r in rects) - pad
    y1 = min(r[1] for r in rects) - pad
    x2 = max(r[2] for r in rects) + pad
    y2 = max(r[3] for r in rects) + pad
    return (max(0, int(x1)), max(0, int(y1)), int(x2), int(y2))


def detect_invoice_fields_on_image(img_bgr, items):
    """Return list of detections {label, rect, text}. Includes field boxes + large block boxes.

    Improvement:
    - totals are often split across boxes ("Total HT:" and "4 475,00 €") -> we pair by geometry
    - table prices are extracted by parsing rows inside `table_block`
    """

    amount_rx = re.compile(r"\b\d{1,3}(?:[\s\u00A0]\d{3})*(?:[\.,]\d{2})\b")

    # --- patterns (tighter to avoid false positives) ---
    patterns = {
        "invoice_number": re.compile(r"\bFAC[-\s]*\d{4}[-\s]*\d+\b", re.I),
        "date_emission": re.compile(r"Date\s*d['’]?émission\s*[:\-]?\s*(\d{1,2}/\d{1,2}/\d{2,4})", re.I),
        "date_echeance": re.compile(r"Date\s*d['’]?échéance\s*[:\-]?\s*(\d{1,2}/\d{1,2}/\d{2,4})", re.I),
        "email": re.compile(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", re.I),
        # French phone (starts with 0)
        "phone": re.compile(r"\b0\d(?:\s\d{2}){4}\b"),
        "siret": re.compile(r"\b\d{3}\s?\d{3}\s?\d{3}\s?\d{5}\b"),
        # IBAN: prefer FR for French invoices to reduce noise
        "iban": re.compile(r"\bFR\d{2}[A-Z0-9\s]{10,}\b", re.I),
        # BIC: ONLY when the OCR line contains the label 'BIC'
        "bic": re.compile(r"\bBIC\b\s*[:\-]?\s*([A-Z0-9]{8,11})\b", re.I),
        # Totals: match label even if amount is not on same box
        "total_ttc": re.compile(r"\bTotal\s*TTC\b", re.I),
        "total_ht": re.compile(r"\bTotal\s*HT\b", re.I),
        "tva": re.compile(r"\bTVA\b", re.I),
    }

    # --- keyword anchors for block detection ---
    def _find_anchor(keyword):
        kw = keyword.upper()
        best = None
        for it in items:
            t = (it.get("text") or "").strip().upper()
            if kw in t:
                r = _box_to_rect(it["box"])
                best = r if best is None else best
        return best

    r_fourn = _find_anchor("FOURNISSEUR")
    r_client = _find_anchor("CLIENT")
    r_details = _find_anchor("DÉTAILS") or _find_anchor("DETAILS")
    r_totaux = _find_anchor("TOTAUX")
    r_coord = _find_anchor("COORDONN")  # COORDONNEES BANCAIRES

    # --- collect raw matches ---
    raw = []
    for it in items:
        t = (it.get("text") or "").strip()
        if not t:
            continue
        rect = _box_to_rect(it["box"])
        score = it.get("score") if it.get("score") is not None else 0.0
        for label, rx in patterns.items():
            if rx.search(t):
                raw.append({"label": label, "rect": rect, "text": t, "score": float(score)})

    # --- keep best per label to avoid duplicates ---
    singletons = {
        "invoice_number",
        "date_emission",
        "date_echeance",
        "email",
        "phone",
        "siret",
        "iban",
        "bic",
        "total_ht",
        "tva",
        "total_ttc",
    }

    best = {}
    dets = []
    for d in raw:
        if d["label"] in singletons:
            # rank: score then text length
            key = (d["score"], len(d["text"]))
            if d["label"] not in best or key > best[d["label"]][0]:
                best[d["label"]] = (key, d)
        else:
            dets.append(d)

    for _, d in best.values():
        dets.append({"label": d["label"], "rect": d["rect"], "text": d["text"]})

    # --- large block boxes (merge OCR boxes between anchors) ---
    h, w = img_bgr.shape[:2]
    item_rects = [(_box_to_rect(it["box"]), (it.get("text") or "").strip()) for it in items if (it.get("text") or "").strip()]

    def rects_in_band(y_top, y_bot, x_left=0, x_right=None):
        if x_right is None:
            x_right = w
        rects = []
        for r, _t in item_rects:
            cx, cy = _rect_center(r)
            if y_top <= cy <= y_bot and x_left <= cx <= x_right:
                rects.append(r)
        return rects

    # Supplier block: between FOURNISSEUR and CLIENT, mostly left side
    if r_fourn and r_client and r_client[1] > r_fourn[1]:
        sup_rects = rects_in_band(r_fourn[1], r_client[1], 0, int(w * 0.7))
        u = _union_rect(sup_rects)
        if u:
            dets.append({"label": "supplier_block", "rect": u, "text": "FOURNISSEUR"})

    # Client block: between CLIENT and DETAILS (capture name+address+CP/Ville, exclude table headers)
    if r_client and (r_details or r_totaux):
        y_top = max(0, r_client[1] - 15)
        y_bot = r_details[1] if r_details else (r_totaux[1] if r_totaux else h)
        if y_bot > y_top:
            stop_kw = ("DÉTAILS", "DETAILS", "PRESTATION", "RÉF", "REF", "DESCRIPTION", "QTE", "QTÉ", "PRIX", "TOTAL")
            cli_rects = []
            for r, t in item_rects:
                cx, cy = _rect_center(r)
                if not (y_top <= cy <= y_bot and 0 <= cx <= int(w * 0.8)):
                    continue
                tu = t.upper()
                if any(k in tu for k in stop_kw):
                    continue
                cli_rects.append(r)
            u = _union_rect(cli_rects)
            if u:
                dets.append({"label": "client_block", "rect": u, "text": "CLIENT"})

    # Table block: between DETAILS and TOTAUX
    if r_details and r_totaux and r_totaux[1] > r_details[1]:
        tab_rects = rects_in_band(r_details[1], r_totaux[1], 0, w)
        u = _union_rect(tab_rects)
        if u:
            dets.append({"label": "table_block", "rect": u, "text": "TABLE"})

    # Totals block: below TOTAUX, right side
    if r_totaux:
        tot_rects = rects_in_band(r_totaux[1], h, int(w * 0.55), w)
        u = _union_rect(tot_rects)
        if u:
            dets.append({"label": "totals_block", "rect": u, "text": "TOTAUX"})

    # Bank block: below COORDONNEES, left side
    if r_coord:
        bank_rects = rects_in_band(r_coord[1], h, 0, int(w * 0.7))
        u = _union_rect(bank_rects)
        if u:
            dets.append({"label": "bank_block", "rect": u, "text": "BANQUE"})

    # --- Pair totals label -> nearest amount on same line (handles split boxes) ---
    def _nearest_amount_right(label_rect):
        lx1, ly1, lx2, ly2 = label_rect
        lcy = (ly1 + ly2) / 2.0
        best = None
        for r, t in item_rects:
            if r[0] <= lx2:
                continue
            cy = (r[1] + r[3]) / 2.0
            if abs(cy - lcy) > 18:  # same line tolerance
                continue
            m = amount_rx.search(t.replace("€", ""))
            if not m:
                continue
            dx = r[0] - lx2
            cand = (dx, r, t)
            if best is None or cand[0] < best[0]:
                best = cand
        return best

    # build map of label rects for totals from detected raw matches
    total_label_rects = {}
    for d in dets:
        if d["label"] in ("total_ht", "tva", "total_ttc"):
            total_label_rects[d["label"]] = d["rect"]

    for key in ("total_ht", "tva", "total_ttc"):
        if key in total_label_rects:
            found = _nearest_amount_right(total_label_rects[key])
            if found:
                _dx, r_amt, t_amt = found
                # store a combined detection with the amount
                dets.append({"label": key + "_value", "rect": r_amt, "text": t_amt})

    # --- Table line-items extraction (prices/qty) inside table_block ---
    table_rect = None
    for d in dets:
        if d["label"] == "table_block":
            table_rect = d["rect"]
            break

    line_items = []
    if table_rect:
        # collect items inside table
        table_items = []
        for r, t in item_rects:
            cx, cy = _rect_center(r)
            if (table_rect[0] <= cx <= table_rect[2]) and (table_rect[1] <= cy <= table_rect[3]):
                table_items.append((r, t))
        # group by rows (y)
        table_items.sort(key=lambda x: (x[0][1], x[0][0]))
        rows = []
        row = []
        last_y = None
        for r, t in table_items:
            y = r[1]
            if last_y is None or abs(y - last_y) <= 18:
                row.append((r, t))
                last_y = y if last_y is None else (last_y * 0.7 + y * 0.3)
            else:
                rows.append(row)
                row = [(r, t)]
                last_y = y
        if row:
            rows.append(row)

        # parse each row: extract ref-like + qty + unit price + line total
        ref_rx = re.compile(r"\b[A-Z]{2,4}[-_]\d{2,4}\b")
        qty_rx = re.compile(r"\b\d+\b")
        for row in rows:
            # skip header rows
            joined = " ".join(t for _r, t in row).upper()
            if any(k in joined for k in ["DESIGNATION", "DESCRIPTION", "PRIX", "TOTAL", "QTE", "QTÉ", "RÉF", "REF"]):
                continue
            # sort by x
            row = sorted(row, key=lambda x: x[0][0])
            texts = [t for _r, t in row]

            ref = ""
            for t in texts:
                m = ref_rx.search(t.upper())
                if m:
                    ref = m.group(0)
                    break

            # amounts in row
            amounts = []
            for r, t in row:
                m = amount_rx.search(t.replace("€", ""))
                if m:
                    amounts.append((r[0], m.group(0)))
            amounts.sort(key=lambda x: x[0])

            # qty: first standalone int near amounts (heuristic)
            qty = ""
            for r, t in row:
                if qty_rx.fullmatch(t.strip()) and len(t.strip()) <= 3:
                    qty = t.strip()
                    break

            unit_price = ""
            line_total = ""
            if len(amounts) >= 1:
                line_total = amounts[-1][1]
            if len(amounts) >= 2:
                unit_price = amounts[-2][1]

            desc_parts = [t for t in texts if t != ref and t != qty and (unit_price not in t) and (line_total not in t)]
            desc = " ".join(desc_parts).strip()

            if ref or desc or unit_price or line_total:
                line_items.append({
                    "ref": ref,
                    "description": desc,
                    "qty": qty,
                    "unit_price": unit_price,
                    "line_total": line_total,
                })

    # expose line_items for later cells
    globals()["line_items"] = line_items

    return dets


def draw_detections(img_bgr, dets):
    out = img_bgr.copy()
    colors = {
        # fields
        "invoice_number": (0, 200, 0),
        "date_emission": (0, 200, 0),
        "date_echeance": (0, 200, 0),
        "email": (180, 0, 180),
        "phone": (180, 0, 180),
        "siret": (0, 165, 255),
        "iban": (0, 165, 255),
        "bic": (0, 165, 255),
        "total_ht": (0, 0, 255),
        "tva": (0, 0, 255),
        "total_ttc": (0, 0, 255),
        # blocks
        "supplier_block": (0, 255, 255),
        "client_block": (255, 255, 0),
        "table_block": (255, 0, 0),
        "totals_block": (0, 0, 255),
        "bank_block": (0, 165, 255),
    }

    for d in dets:
        x1, y1, x2, y2 = d["rect"]
        c = colors.get(d["label"], (255, 255, 0))
        thick = 4 if d["label"].endswith("_block") else 2
        cv2.rectangle(out, (x1, y1), (x2, y2), c, thick)
        cv2.putText(out, d["label"], (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, c, 2)

    return out


# --- Run on the current example image ---
# (Downscale before OCR to avoid kernel crashes on large images)
paddle_ocr = globals().get("paddle_ocr", None)
FACTURE_EXAMPLE = globals().get("FACTURE_EXAMPLE", None)

if paddle_ocr is None:
    print("PaddleOCR non initialisé. Exécute la cellule d'initialisation PaddleOCR.")
elif FACTURE_EXAMPLE is None:
    print("FACTURE_EXAMPLE est None. Exécute la cellule de sélection d'image.")
else:
    cv2.setNumThreads(1)
    img_bgr_full = cv2.imread(str(FACTURE_EXAMPLE))
    if img_bgr_full is None:
        raise FileNotFoundError(f"Cannot read image: {FACTURE_EXAMPLE}")

    h0, w0 = img_bgr_full.shape[:2]
    max_side = 1280  # safe default
    scale = min(1.0, max_side / float(max(h0, w0)))
    if scale < 1.0:
        img_bgr = cv2.resize(img_bgr_full, (int(w0 * scale), int(h0 * scale)), interpolation=cv2.INTER_AREA)
    else:
        img_bgr = img_bgr_full

    # PaddleOCR on resized image
    res = paddle_ocr.predict(img_bgr)
    items = _paddle_predict_to_items(res)

    dets = detect_invoice_fields_on_image(img_bgr, items)

    # Rescale rects back to full image coordinates (for drawing on full-res)
    if scale < 1.0:
        inv = 1.0 / scale
        for d in dets:
            x1, y1, x2, y2 = d["rect"]
            d["rect"] = (int(x1 * inv), int(y1 * inv), int(x2 * inv), int(y2 * inv))

    annotated = draw_detections(img_bgr_full, dets)

    print("Detected fields:", len(dets))
    for d in dets:
        if d['label'] in ('invoice_number','date_emission','date_echeance','phone','email','siret','tva','total_ht_value','tva_value','total_ttc_value'):
            print('-', d['label'], '->', d['text'])

    # Show extracted table line items (first 5)
    line_items = globals().get('line_items', [])
    print("Line items extracted:", len(line_items))
    for it in line_items[:5]:
        print('  -', it)

    plt.figure(figsize=(12, 16))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Champs détectés sur l'image (PaddleOCR + bbox)")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

Aucune image facture trouvée dans /Users/aymanehajli/Desktop/IPSSI/M2/HACKATHON/ocr traitement/facture-images
FACTURE_EXAMPLE est None. Exécute la cellule de sélection d'image.


### Extraction complète (depuis les bbox) + export CSV

Cette cellule :
- reconstruit le texte dans chaque **bloc** (FOURNISSEUR / CLIENT / TABLE / TOTAUX / BANQUE)
- extrait les champs importants (n° facture, dates, fournisseur, client, totaux, SIRET/TVA/IBAN/BIC…)
- exporte en CSV (1 ligne par facture)

In [18]:
import re
import csv

# Expect these from the direct-detection cell
# - items: list of {box, text, score}
# - dets: list of {label, rect, text}

items = globals().get("items", None)
dets = globals().get("dets", None)
FACTURE_EXAMPLE = globals().get("FACTURE_EXAMPLE", None)

if not items or not dets:
    print("Run the direct detection cell first (it defines `items` and `dets`).")
else:
    def rect_center(r):
        x1, y1, x2, y2 = r
        return ((x1+x2)/2.0, (y1+y2)/2.0)

    def inside(rect, r):
        x, y = rect_center(r)
        x1,y1,x2,y2 = rect
        return (x1 <= x <= x2) and (y1 <= y <= y2)

    def box_to_rect(box):
        import numpy as np
        b = np.array(box)
        return (int(b[:,0].min()), int(b[:,1].min()), int(b[:,0].max()), int(b[:,1].max()))

    # Build map label->rect for blocks
    blocks = {d["label"]: d["rect"] for d in dets if d["label"].endswith("_block")}

    def text_in_block(block_label):
        rect = blocks.get(block_label)
        if not rect:
            return ""
        rows = []
        for it in items:
            t = (it.get("text") or "").strip()
            if not t:
                continue
            r = box_to_rect(it["box"])
            if inside(rect, r):
                rows.append((r[1], r[0], t))  # sort by y then x
        rows.sort()
        return "\n".join([t for _,_,t in rows])

    supplier_text = text_in_block("supplier_block")
    client_text = text_in_block("client_block")
    table_text = text_in_block("table_block")
    totals_text = text_in_block("totals_block")
    bank_text = text_in_block("bank_block")

    # Field-level values from dets (best per label already)
    field_map = {}
    for d in dets:
        if d["label"].endswith("_block"):
            continue
        field_map[d["label"]] = d["text"]

    def first_match(rx, text):
        m = re.search(rx, text, re.I)
        return m.group(1).strip() if m else ""

    # Parse supplier/client basic identity from reconstructed block text
    def parse_party(block_text, keyword):
        lines = [l.strip() for l in (block_text or "").split("\n") if l.strip()]
        # remove the keyword line itself if present
        lines = [l for l in lines if l.upper() != keyword]

        # drop noisy headers if they slipped in
        drop = (
            "DÉTAILS", "DETAILS", "PRESTATIONS", "PRESTATION", "RÉF", "REF",
            "DESCRIPTION", "QTÉ", "QTE", "PRIX", "TOTAL"
        )
        lines = [l for l in lines if not any(k in l.upper() for k in drop)]

        name = lines[0] if len(lines) >= 1 else ""
        address = lines[1] if len(lines) >= 2 else ""

        # city line: prefer French pattern "75008 PARIS" / "69002 LYON"
        city = ""
        for l in lines[2:6]:
            if re.search(r"\b\d{5}\b", l):
                city = l
                break
        if not city:
            city = lines[2] if len(lines) >= 3 else ""

        return name, address, city

    supplier_name, supplier_address, supplier_city = parse_party(supplier_text, "FOURNISSEUR")
    client_name, client_address, client_city = parse_party(client_text, "CLIENT")

    # Parse totals from totals_text (often better than single-line matches)
    total_ht = first_match(r"Total\s*HT\s*:?\s*([\d\s.,]+)\s*€?", totals_text)
    tva_amt = first_match(r"TVA\s*:?\s*([\d\s.,]+)\s*€?", totals_text)
    total_ttc = first_match(r"Total\s*TTC\s*:?\s*([\d\s.,]+)\s*€?", totals_text)

    # If not found in totals_text, fallback to field_map values
    if not total_ht:
        total_ht = first_match(r"Total\s*HT\s*:?\s*([\d\s.,]+)", field_map.get("total_ht", ""))
    if not tva_amt:
        tva_amt = first_match(r"TVA\s*:?\s*([\d\s.,]+)", field_map.get("tva", ""))
    if not total_ttc:
        total_ttc = first_match(r"Total\s*TTC\s*:?\s*([\d\s.,]+)", field_map.get("total_ttc", ""))

    # Extract IBAN/BIC from bank block
    iban = first_match(r"\bIBAN\b\s*:?\s*([A-Z0-9\s]{10,})", bank_text)
    bic = first_match(r"\bBIC\b\s*:?\s*([A-Z0-9]{8,11})", bank_text)

    # Invoice number and dates already detected
    invoice_number = first_match(r"(FAC[-\s]*\d{4}[-\s]*\d+)", field_map.get("invoice_number", ""))
    date_emission = first_match(r"(\d{1,2}/\d{1,2}/\d{2,4})", field_map.get("date_emission", ""))
    date_echeance = first_match(r"(\d{1,2}/\d{1,2}/\d{2,4})", field_map.get("date_echeance", ""))

    phone = first_match(r"(0\d(?:\s\d{2}){4})", field_map.get("phone", ""))
    email = first_match(r"([A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,})", field_map.get("email", ""))
    siret = first_match(r"(\d{3}\s?\d{3}\s?\d{3}\s?\d{5})", field_map.get("siret", ""))
    tva_id = field_map.get("tva", "")

    row = {
        "image_file": str(FACTURE_EXAMPLE) if FACTURE_EXAMPLE else "",
        "invoice_number": invoice_number,
        "date_emission": date_emission,
        "date_echeance": date_echeance,
        "supplier_name": supplier_name,
        "supplier_address": supplier_address,
        "supplier_city": supplier_city,
        "supplier_phone": phone,
        "supplier_email": email,
        "supplier_siret": re.sub(r"\s+", "", siret),
        "supplier_tva": tva_id,
        "client_name": client_name,
        "client_address": client_address,
        "client_city": client_city,
        "total_ht": total_ht,
        "tva": tva_amt,
        "total_ttc": total_ttc,
        "bic": bic,
        "iban": re.sub(r"\s+", " ", iban).strip(),
        "raw_supplier_block": supplier_text,
        "raw_client_block": client_text,
        "raw_totals_block": totals_text,
    }

    # Affichage simple
    for k, v in row.items():
        print(f"{k}: {v}")

    out_csv = "paddleocr_invoice_fields.csv"
    with open(out_csv, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()), delimiter=";")
        w.writeheader()
        w.writerow(row)

    print("Saved:", out_csv)

Run the direct detection cell first (it defines `items` and `dets`).


## Validation intelligente & détection d’incohérences (facture vs attestation)

Objectif : détecter automatiquement des incohérences entre **la facture** (OCR) et une **attestation de vigilance** (ou autre document de conformité).

### Exemples de règles
- **SIRET différent** entre la facture et l’attestation
- **TVA incohérente** (format / valeur / mismatch)
- **Attestation expirée** (date de validité dépassée)
- **Montants incohérents** (si on a HT/TVA/TTC)

### Sortie
- une liste d’issues (règles)
- un **score** (rule-based) + optionnellement un score via un modèle d’**anomaly detection** (si dispo)


In [19]:
import re
from dataclasses import dataclass
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Tuple


def _only_digits(s: str) -> str:
    return re.sub(r"\D+", "", s or "")


def normalize_siret(s: str) -> str:
    """SIRET attendu: 14 chiffres."""
    d = _only_digits(s)
    return d[:14] if len(d) >= 14 else d


def luhn_is_valid(number: str) -> bool:
    """Validation Luhn (SIREN/SIRET)."""
    digits = [int(x) for x in _only_digits(number)]
    if not digits:
        return False
    checksum = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        checksum += d
    return checksum % 10 == 0


def normalize_tva_fr(tva: str) -> str:
    """Normalise TVA FR: 'FR' + 2 clés + 9 chiffres (souvent SIREN).

    On garde un format compact sans espaces. Exemple: 'FRXX123456789'.
    """
    t = (tva or "").upper().replace(" ", "")
    t = t.replace("TVA", "").replace(":", "")
    m = re.search(r"\bFR\w{2}\d{9}\b", t)
    return m.group(0) if m else t


def parse_date_fr(s: str) -> Optional[date]:
    """Parse date FR la plus commune: JJ/MM/AAAA ou JJ/MM/AA."""
    s = (s or "").strip()
    m = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", s)
    if not m:
        return None
    dd, mm, yy = int(m.group(1)), int(m.group(2)), int(m.group(3))
    if yy < 100:
        yy += 2000
    try:
        return date(yy, mm, dd)
    except Exception:
        return None


def parse_amount_fr(s: str) -> Optional[float]:
    """Parse montant type '8 000,00' ou '800,00 €' -> float."""
    if not s:
        return None
    txt = s
    txt = txt.replace("€", "").replace("\u00A0", " ")
    txt = txt.strip()
    m = re.search(r"\b\d{1,3}(?:[\s]\d{3})*(?:[\.,]\d{2})\b", txt)
    if not m:
        return None
    x = m.group(0)
    x = x.replace(" ", "")
    x = x.replace(",", ".")
    try:
        return float(x)
    except Exception:
        return None


@dataclass
class ValidationIssue:
    code: str
    severity: str  # 'low'|'medium'|'high'
    message: str


def validate_facture_vs_attestation(
    facture: Dict[str, Any],
    attestation: Dict[str, Any],
    today: Optional[date] = None,
) -> Tuple[List[ValidationIssue], float]:
    """Rule-based validation + score.

    `facture`: dict issu de la cellule d'export (invoice_number, supplier_siret, supplier_tva, date_emission, total_ht, tva, total_ttc, ...)
    `attestation`: dict minimal (siret, tva, valid_until)

    Retour:
    - issues
    - score (0..1) : plus haut = plus suspect
    """
    today = today or date.today()

    issues: List[ValidationIssue] = []
    score = 0.0

    f_siret = normalize_siret(str(facture.get("supplier_siret", "")))
    a_siret = normalize_siret(str(attestation.get("siret", "")))

    if f_siret and len(f_siret) == 14 and not luhn_is_valid(f_siret):
        issues.append(ValidationIssue("SIRET_INVALID", "high", f"SIRET facture invalide (Luhn): {f_siret}"))
        score += 0.35

    if a_siret and len(a_siret) == 14 and not luhn_is_valid(a_siret):
        issues.append(ValidationIssue("ATTEST_SIRET_INVALID", "high", f"SIRET attestation invalide (Luhn): {a_siret}"))
        score += 0.35

    if f_siret and a_siret and f_siret != a_siret:
        issues.append(ValidationIssue("SIRET_MISMATCH", "high", f"SIRET différent (facture={f_siret}, attestation={a_siret})"))
        score += 0.6

    f_tva = normalize_tva_fr(str(facture.get("supplier_tva", "")))
    a_tva = normalize_tva_fr(str(attestation.get("tva", "")))

    # Format TVA FR attendu
    if f_tva and not re.search(r"\bFR\w{2}\d{9}\b", f_tva):
        issues.append(ValidationIssue("TVA_FORMAT", "medium", f"TVA facture format inattendu: '{f_tva}'"))
        score += 0.2

    if a_tva and not re.search(r"\bFR\w{2}\d{9}\b", a_tva):
        issues.append(ValidationIssue("ATTEST_TVA_FORMAT", "medium", f"TVA attestation format inattendu: '{a_tva}'"))
        score += 0.2

    if f_tva and a_tva and f_tva.startswith("FR") and a_tva.startswith("FR") and f_tva != a_tva:
        issues.append(ValidationIssue("TVA_MISMATCH", "high", f"TVA différente (facture={f_tva}, attestation={a_tva})"))
        score += 0.45

    # Expiration attestation
    valid_until = attestation.get("valid_until") or attestation.get("date_expiration") or attestation.get("date_validite")
    d_until = parse_date_fr(str(valid_until)) if valid_until else None
    if d_until is None and valid_until:
        issues.append(ValidationIssue("ATTEST_DATE_PARSE", "low", f"Date attestation non parsable: '{valid_until}'"))
        score += 0.05
    if d_until and d_until < today:
        issues.append(ValidationIssue("ATTEST_EXPIRED", "high", f"Attestation expirée (valid_until={d_until.isoformat()})"))
        score += 0.7

    # Cohérence montants: TTC ≈ HT + TVA
    ht = parse_amount_fr(str(facture.get("total_ht", "")))
    tva_amt = parse_amount_fr(str(facture.get("tva", "")))
    ttc = parse_amount_fr(str(facture.get("total_ttc", "")))
    if ht is not None and tva_amt is not None and ttc is not None:
        if abs((ht + tva_amt) - ttc) > 0.05:
            issues.append(ValidationIssue("TOTALS_INCOHERENT", "medium", f"Incohérence totaux: HT({ht}) + TVA({tva_amt}) != TTC({ttc})"))
            score += 0.25

    # Clamp score
    score = max(0.0, min(1.0, score))
    return issues, score


# ---- Exemple d'utilisation (à adapter quand tu auras l'OCR attestation) ----
# La facture vient de la cellule précédente, via `row`.
row = globals().get("row")
if not isinstance(row, dict):
    print("La variable `row` n'existe pas. Exécute d'abord la cellule d'export CSV facture.")
else:
    # Exemple attestation (mock) : remplace par la sortie OCR de l'attestation.
    att = {
        "siret": row.get("supplier_siret", ""),
        "tva": row.get("supplier_tva", ""),
        "valid_until": "31/12/2026",
    }

    issues, score = validate_facture_vs_attestation(row, att)
    print("Score suspicion:", score)
    if not issues:
        print("Aucune incohérence détectée (règles).")
    else:
        for it in issues:
            print(f"- [{it.severity}] {it.code}: {it.message}")


La variable `row` n'existe pas. Exécute d'abord la cellule d'export CSV facture.


In [20]:
# --- Optionnel: anomaly detection (si scikit-learn est disponible) ---
# Idée: apprendre un "profil normal" sur un historique de factures/attestations.
# Ici on montre la forme: vecteur de features -> score d'anomalie.

from typing import Iterable


def build_features(facture: Dict[str, Any], attestation: Dict[str, Any]) -> Dict[str, float]:
    f_siret = normalize_siret(str(facture.get("supplier_siret", "")))
    a_siret = normalize_siret(str(attestation.get("siret", "")))
    f_tva = normalize_tva_fr(str(facture.get("supplier_tva", "")))
    a_tva = normalize_tva_fr(str(attestation.get("tva", "")))

    ht = parse_amount_fr(str(facture.get("total_ht", "")))
    tva_amt = parse_amount_fr(str(facture.get("tva", "")))
    ttc = parse_amount_fr(str(facture.get("total_ttc", "")))

    valid_until = attestation.get("valid_until") or attestation.get("date_expiration") or attestation.get("date_validite")
    d_until = parse_date_fr(str(valid_until)) if valid_until else None

    # Features simples (0/1 ou numériques)
    feats = {
        "siret_match": 1.0 if (f_siret and a_siret and f_siret == a_siret) else 0.0,
        "siret_facture_luhn": 1.0 if (len(f_siret) == 14 and luhn_is_valid(f_siret)) else 0.0,
        "tva_match": 1.0 if (f_tva.startswith("FR") and a_tva.startswith("FR") and f_tva == a_tva) else 0.0,
        "has_amounts": 1.0 if (ht is not None and tva_amt is not None and ttc is not None) else 0.0,
        "amounts_delta": float(abs((ht + tva_amt) - ttc)) if (ht is not None and tva_amt is not None and ttc is not None) else 0.0,
        "attestation_days_left": float((d_until - date.today()).days) if d_until else 0.0,
    }
    return feats


def try_anomaly_detection(train_rows: Iterable[Tuple[Dict[str, Any], Dict[str, Any]]], test_row: Tuple[Dict[str, Any], Dict[str, Any]]):
    """Retourne un score d'anomalie (plus haut = plus anormal) ou None si sklearn absent."""
    try:
        import numpy as np
        from sklearn.ensemble import IsolationForest  # type: ignore
    except Exception:
        return None

    def vec(d: Dict[str, float], keys: List[str]) -> List[float]:
        return [float(d.get(k, 0.0)) for k in keys]

    train_feats = [build_features(f, a) for (f, a) in train_rows]
    if not train_feats:
        return None

    keys = sorted(train_feats[0].keys())
    X = np.array([vec(d, keys) for d in train_feats], dtype=float)

    model = IsolationForest(n_estimators=200, random_state=42, contamination=0.05)
    model.fit(X)

    test_feats = build_features(test_row[0], test_row[1])
    x = np.array([vec(test_feats, keys)], dtype=float)

    # decision_function: plus grand = plus normal. On inverse pour "anomaly_score".
    normality = float(model.decision_function(x)[0])
    anomaly_score = -normality
    return anomaly_score, test_feats


# Démo (placeholder): on simule un mini "train" avec variations.
row = globals().get("row")
if isinstance(row, dict):
    base_att = {"siret": row.get("supplier_siret", ""), "tva": row.get("supplier_tva", ""), "valid_until": "31/12/2026"}
    train = [(row, base_att)] * 20

    # test: mismatch SIRET
    test_att = {**base_att, "siret": "12345678901234"}
    out = try_anomaly_detection(train, (row, test_att))
    if out is None:
        print("[anomaly detection] scikit-learn non installé -> skip (OK).")
    else:
        anomaly_score, feats = out
        print("[anomaly detection] score:", anomaly_score)
        print("features:", feats)


## Vérification INSEE SIRENE (API) — infos importantes

Cette section interroge l’API INSEE SIRENE (avec `X-INSEE-Api-Key-Integration`) pour récupérer des infos fiables à partir d’un **SIREN** (ou **SIRET**).

### Pré-requis
- Avoir la variable d’env : `INSEE_API_KEY_INTEGRATION`
- (Optionnel) Forcer la base URL : `INSEE_SIRENE_BASE_URL` (ex: `https://api.insee.fr/api-sirene/3.11`)


In [21]:
import os
import json
import urllib.request
import urllib.error
import re
from datetime import date
from typing import Any, Dict, Optional


def _digits(s: str) -> str:
    return re.sub(r"\D+", "", s or "")


def _http_get_json(url: str, api_key: str) -> Dict[str, Any]:
    req = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "X-INSEE-Api-Key-Integration": api_key,
        },
        method="GET",
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace") if hasattr(e, "read") else ""
        return {"_http_status": int(getattr(e, "code", 0) or 0), "_error": body, "_url": url}


def extract_unite_legale_summary(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Extrait les infos importantes depuis une réponse /siren/{siren}."""
    ul = payload.get("uniteLegale") or payload  # parfois ul directement
    periods = ul.get("periodesUniteLegale") or []
    # La dernière période est généralement la plus récente
    last = periods[-1] if isinstance(periods, list) and periods else {}

    denom = last.get("denominationUniteLegale") or last.get("nomUniteLegale") or ""
    denom_usuelle = (
        last.get("denominationUsuelle1UniteLegale")
        or last.get("denominationUsuelle2UniteLegale")
        or last.get("denominationUsuelle3UniteLegale")
        or ""
    )

    return {
        "siren": ul.get("siren"),
        "denomination": denom,
        "denomination_usuelle": denom_usuelle,
        "date_creation": ul.get("dateCreationUniteLegale"),
        "etat_administratif": last.get("etatAdministratifUniteLegale"),
        "activite_principale": last.get("activitePrincipaleUniteLegale") or ul.get("activitePrincipaleUniteLegale"),
        "categorie_juridique": last.get("categorieJuridiqueUniteLegale"),
        "nic_siege": last.get("nicSiegeUniteLegale"),
        "statut_diffusion": ul.get("statutDiffusionUniteLegale"),
        "date_dernier_traitement": ul.get("dateDernierTraitementUniteLegale"),
    }


def extract_etablissement_summary(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Extrait les infos importantes depuis une réponse /siret/{siret}."""
    etab = payload.get("etablissement") or payload
    periodes = etab.get("periodesEtablissement") or []
    last = periodes[-1] if isinstance(periodes, list) and periodes else {}

    addr = etab.get("adresseEtablissement") or {}
    libelle_voie = " ".join(
        [
            str(addr.get("numeroVoieEtablissement") or "").strip(),
            str(addr.get("typeVoieEtablissement") or "").strip(),
            str(addr.get("libelleVoieEtablissement") or "").strip(),
        ]
    ).strip()

    return {
        "siret": etab.get("siret"),
        "siren": etab.get("siren"),
        "etat_administratif_etab": last.get("etatAdministratifEtablissement"),
        "date_creation_etab": etab.get("dateCreationEtablissement"),
        "activite_principale_etab": last.get("activitePrincipaleEtablissement"),
        "enseigne": last.get("enseigne1Etablissement") or last.get("enseigne2Etablissement") or last.get("enseigne3Etablissement"),
        "adresse": libelle_voie,
        "code_postal": addr.get("codePostalEtablissement"),
        "commune": addr.get("libelleCommuneEtablissement"),
    }


def insee_lookup(identifier: str, base_url: Optional[str] = None) -> Dict[str, Any]:
    """Lookup INSEE pour SIREN (9) ou SIRET (14) et retourne un résumé."""
    api_key = os.environ.get("INSEE_API_KEY_INTEGRATION", "").strip()
    if not api_key:
        return {"ok": False, "error": "Missing env var INSEE_API_KEY_INTEGRATION"}

    base_url = (base_url or os.environ.get("INSEE_SIRENE_BASE_URL", "").strip() or "https://api.insee.fr/api-sirene/3.11").rstrip("/")

    d = _digits(identifier)
    if len(d) == 9:
        url = f"{base_url}/siren/{d}"
        payload = _http_get_json(url, api_key)
        if payload.get("_http_status"):
            return {"ok": False, "status": payload.get("_http_status"), "details": payload.get("_error"), "url": payload.get("_url")}
        return {"ok": True, "kind": "siren", "summary": extract_unite_legale_summary(payload)}

    if len(d) == 14:
        url = f"{base_url}/siret/{d}"
        payload = _http_get_json(url, api_key)
        if payload.get("_http_status"):
            return {"ok": False, "status": payload.get("_http_status"), "details": payload.get("_error"), "url": payload.get("_url")}
        # Bonus: on récupère aussi l'unité légale (si possible)
        etab_sum = extract_etablissement_summary(payload)
        ul_sum = {}
        try:
            url_ul = f"{base_url}/siren/{etab_sum.get('siren')}"
            ul_payload = _http_get_json(url_ul, api_key)
            if not ul_payload.get("_http_status"):
                ul_sum = extract_unite_legale_summary(ul_payload)
        except Exception:
            pass
        return {"ok": True, "kind": "siret", "summary": {**etab_sum, **{f"ul_{k}": v for k, v in ul_sum.items()}}}

    return {"ok": False, "error": f"Identifier must be SIREN(9) or SIRET(14), got {len(d)} digits"}


# ==== EXEMPLE ====
# Mets ici ton SIREN/SIRET
TEST_ID = "309634954"

res = insee_lookup(TEST_ID)
print(json.dumps(res, ensure_ascii=False, indent=2))

if res.get("ok"):
    print("\n--- Infos importantes ---")
    for k, v in (res.get("summary") or {}).items():
        print(f"{k}: {v}")


{
  "ok": true,
  "kind": "siren",
  "summary": {
    "siren": "309634954",
    "denomination": "REPARATION ENTRETIEN ET MAINTENANCE",
    "denomination_usuelle": "",
    "date_creation": "1977-12-25",
    "etat_administratif": "C",
    "activite_principale": "81.21",
    "categorie_juridique": "5499",
    "nic_siege": "00015",
    "statut_diffusion": "P",
    "date_dernier_traitement": "2024-03-22T14:26:06.001"
  }
}

--- Infos importantes ---
siren: 309634954
denomination: REPARATION ENTRETIEN ET MAINTENANCE
denomination_usuelle: 
date_creation: 1977-12-25
etat_administratif: C
activite_principale: 81.21
categorie_juridique: 5499
nic_siege: 00015
statut_diffusion: P
date_dernier_traitement: 2024-03-22T14:26:06.001
